<a href="https://colab.research.google.com/github/NabilBADRI/Competition-StanceNakba-2026/blob/main/Baseline_StanceNakba_subtasks_A_and_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Subtask A**

In [ ]:
"""
==============================================================================
BASELINE FOR STANCE DETECTION (Subtask A)
==============================================================================
Standard DeBERTa-v3-base fine-tuning baseline for shared task
Model: microsoft/deberta-v3-base
Task: 3-class stance classification (Neutral, Pro-Israel, Pro-Palestine)

==============================================================================
"""

# ==============================================================================
# INSTALLATION
# ==============================================================================
!pip install -q transformers datasets evaluate scikit-learn accelerate seaborn

# ==============================================================================
# IMPORTS
# ==============================================================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
SEED = 42
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")



Device: cuda


In [ ]:
# ==============================================================================
# PART 1: TRAINING
# ==============================================================================
print("\n" + "=" * 70)
print("PART 1: TRAINING")
print("=" * 70)
print("Upload labeled training dataset")

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

# Prepare data

print(f"\nDataset: {len(df)} samples")
print(f"Class distribution:")
print(df['label'].value_counts().sort_index())

label_list = ["Neutral", "Pro-Israel", "Pro-Palestine"]
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

# Validate labels
extra_labels = sorted(set(df["label"].unique()) - set(label_list))
missing_labels = [l for l in label_list if l not in set(df["label"].unique())]

if missing_labels:
    print("\nWarning: Missing labels in the uploaded files:", missing_labels)
if extra_labels:
    raise ValueError(f"Unknown stance labels found in data: {extra_labels}")

print("\nLabel mappings:")
for label, idx in label2id.items():
    print(f"  {label:13} → {idx}")

# Map labels for each split
df["label"] = df["label"].map(label2id).astype(int)


# Split
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['label'])
dev_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

print(f"\nTrain: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

# Tokenization
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding='max_length')

train_dataset = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_function, batched=True)
dev_dataset = Dataset.from_pandas(dev_df[['text', 'label']]).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_function, batched=True)

print("✓ Tokenization complete")

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(device)

print(f"✓ Model loaded: {MODEL_NAME}")

# Training args
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

# Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    return {'accuracy': accuracy, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Train
print("\n" + "=" * 70)
print("Training...")
print("=" * 70)
trainer.train()
print("✓ Training complete!")

# Evaluate
print("\n" + "=" * 70)
print("Test Set Results")
print("=" * 70)
test_results = trainer.evaluate(test_dataset)
print(f"Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']:.2%})")
print(f"F1 Macro: {test_results['eval_f1_macro']:.4f}")

# Save model
trainer.save_model('./trained_model')
tokenizer.save_pretrained('./trained_model')
print("\n✓ Model saved to: ./trained_model")


PART 1: TRAINING
Upload labeled training dataset


Saving Subtask_A_train.csv to Subtask_A_train (2).csv

Dataset: 980 samples
Class distribution:
label
Neutral          327
Pro-Israel       327
Pro-Palestine    326
Name: count, dtype: int64

Label mappings:
  Neutral       → 0
  Pro-Israel    → 1
  Pro-Palestine → 2

Train: 784 | Dev: 98 | Test: 98


Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded: microsoft/deberta-v3-base

Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,1.097826,0.367347,0.239801,0.236850
2,1.101500,0.980362,0.540816,0.530171,0.529301
3,1.020100,0.787724,0.693878,0.687831,0.687062
4,0.752900,0.753028,0.714286,0.706360,0.705404


✓ Training complete!

Test Set Results


Accuracy: 0.8265 (82.65%)
F1 Macro: 0.8223

✓ Model saved to: ./trained_model


In [ ]:

# ==============================================================================
# PART 2: PREDICTION ON UNLABELED DATA
# ==============================================================================
print("\n" + "=" * 70)
print("PART 2: PREDICTION ON UNLABELED DATA")
print("=" * 70)
print("Upload unlabeled dev/test file (with 'ID' and 'text')")

uploaded_unlabeled = files.upload()
df_unlabeled = pd.read_csv(list(uploaded_unlabeled.keys())[0])

print(f"\n✓ Loaded: {len(df_unlabeled)} unlabeled samples")
print(f"Columns: {df_unlabeled.columns.tolist()}")

# Verify columns
if 'id' not in df_unlabeled.columns or 'text' not in df_unlabeled.columns:
    raise ValueError("Unlabeled file must have 'Document ID' and 'text_clean' columns")

# Make predictions
print("\nMaking predictions...")
model.eval()

def predict_batch(texts, batch_size=16):
    all_predictions = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)
            all_predictions.extend(predictions.cpu().numpy())
    return all_predictions

texts = df_unlabeled['text'].tolist()
predicted_labels_ids = predict_batch(texts)
predicted_labels = [id2label[int(pred_id)] for pred_id in predicted_labels_ids]

print(f"✓ Predictions complete!")
print(f"\nPredicted distribution:")
print(pd.Series(predicted_labels).value_counts())





PART 2: PREDICTION ON UNLABELED DATA
Upload unlabeled dev/test file (with 'ID' and 'text')


Saving Subtask_A_val.csv to Subtask_A_val (2).csv

✓ Loaded: 210 unlabeled samples
Columns: ['id', 'text']

Making predictions...
✓ Predictions complete!

Predicted distribution:
Pro-Israel       88
Neutral          67
Pro-Palestine    55
Name: count, dtype: int64


In [ ]:
# ==============================================================================
# PART 3: SAVE PREDICTIONS
# ==============================================================================
import zipfile

print("\n" + "=" * 70)
print("PART 3: SAVING PREDICTIONS")
print("=" * 70)

# Create predictions dataframe
predictions_df = pd.DataFrame({
    'id': df_unlabeled['id'],
    'text': df_unlabeled['text'],
    'prediction': predicted_labels
})

# Save CSV
output_csv = 'predictions.csv'
predictions_df.to_csv(output_csv, index=False)
print(f"✓ Saved: {output_csv}")

# Create ZIP
output_zip = 'predictions.zip'
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(output_csv, arcname=output_csv)
print(f"✓ Created: {output_zip}")

# Display sample predictions
print("\nFirst 10 predictions:")
print(predictions_df.head(10))

# Download files
print("\n" + "=" * 70)
print("Downloading results...")
print("=" * 70)



PART 3: SAVING PREDICTIONS
✓ Saved: predictions.csv
✓ Created: predictions.zip

First 10 predictions:
    id                                               text     prediction
0  981  "" because being anti genocide is good. "" mea...  Pro-Palestine
1  982  One year of Genocide The cries of Al-Aqsa echo...        Neutral
2  983  may god come in revenge plestine i will not be...  Pro-Palestine
3  984  I live in a country that is publicly against I...     Pro-Israel
4  985  I pray that our God will give her peace. Such ...     Pro-Israel
5  986  Typical Zionists. . I would offer my home as a...  Pro-Palestine
6  987  Follow '' if you want to see the reality of Is...        Neutral
7  988  How do you as a Christian shout when your olde...  Pro-Palestine
8  989  No worries Omar, there are more than 20000 syr...     Pro-Israel
9  990  . I STAND WITH GOD'S PEOPLE. THEY WILL EXTERMI...     Pro-Israel



In [ ]:
files.download(output_csv)
files.download(output_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# SUMMARY
# ==============================================================================
print("\n" + "=" * 70)
print("✓ COMPLETE!")
print("=" * 70)
print(f"\nTraining Results:")
print(f"  Test Accuracy: {test_results['eval_accuracy']:.2%}")
print(f"  Test F1 Macro: {test_results['eval_f1_macro']:.4f}")
print(f"\nPredictions:")
print(f"  Samples: {len(predictions_df)}")
print(f"  Files: predictions.csv, predictions.zip")
print("\n" + "=" * 70)


✓ COMPLETE!

Training Results:
  Test Accuracy: 82.65%
  Test F1 Macro: 0.8223

Predictions:
  Samples: 210
  Files: predictions.csv, predictions.zip



# **Subtask B**

In [ ]:
!pip install transformers torch pandas scikit-learn openpyxl

In [ ]:
"""
==============================================================================
BASELINE FOR ARABIC STANCE DETECTION (Subtask B)
==============================================================================
AraBERT v2 fine-tuning baseline for Arabic stance classification

Model: aubmindlab/bert-base-arabertv02 (AraBERT v2)
Task: 3-class stance classification (Pro, Against, Neutral)
Language: Arabic

Training Setup:
- Training Set: Used for model training (with labels)
- Evaluation Set: Used for prediction (without labels)
- Test Set: Not used in this version

Output:
- evaluation_predictions.csv: Predictions with 'prediction' column
- evaluation_predictions.zip: Compressed predictions file

Requirements:
- Google Colab with T4 GPU (recommended)
- Training time: ~5-15 minutes with GPU

==============================================================================
"""

#%% ============== CONFIG - UPDATE THESE PATHS ==============
"""
IMPORTANT: Enable GPU Runtime in Google Colab for faster training!
----------------------------------------------------------------------
1. Go to: Runtime > Change runtime type
2. Select: T4 GPU as Hardware accelerator
3. Click: Save
"""

# Data paths - UPDATE THESE!
TRAIN_PATH = "Subtask_B_train.csv"
EVAL_PATH = "Subtask_B_eval.csv"

# Model settings
MODEL_NAME = "aubmindlab/bert-base-arabertv02"  # AraBERT v2
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
OUTPUT_DIR = "./stance_model"

In [ ]:
#Do not change this
# Output paths
PREDICTIONS_CSV = "predictions.csv"
PREDICTIONS_ZIP = "predictions.zip"

# Label mapping
LABEL2ID = {"pro": 0, "against": 1, "neutral": 2}
ID2LABEL = {0: "pro", 1: "against", 2: "neutral"}
NUM_LABELS = 3

In [ ]:
#%% ============== IMPORTS ==============
import os
import re
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import warnings
import zipfile
from google.colab import files
warnings.filterwarnings('ignore')

#set seeds
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


CUDA available: True
Device: Tesla T4


In [ ]:
#%% ============== TEXT PREPROCESSING ==============
def normalize_arabic(text):
    """Basic Arabic text normalization"""
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # Normalize alef variants
    text = re.sub(r'[إأآا]', 'ا', text)

    # Normalize alef maqsura to yaa
    text = re.sub(r'ى', 'ي', text)

    # Normalize taa marbuta to haa
    text = re.sub(r'ة', 'ه', text)

    # Remove tatweel
    text = re.sub(r'ـ', '', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove URLs and mentions
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)

    return text.strip()

In [ ]:
#%% ============== DATASET CLASS ==============
class StanceDataset(Dataset):
    def __init__(self, texts, topics, labels, tokenizer, max_length):
        self.texts = texts
        self.topics = topics
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        topic = str(self.topics[idx])

        encoding = self.tokenizer(
            topic,
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].flatten()

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [ ]:
#%% ============== LOAD DATA ==============
def load_data(file_path, has_labels=True):
    """Load data from CSV or Excel"""
    df = pd.read_csv(file_path)
    df['Sentence_norm'] = df['sentence'].apply(normalize_arabic)
    df['Topic_norm'] = df['topic'].apply(normalize_arabic)
    return df

# Load training and evaluation data
print("Loading data...")
train_df = load_data(TRAIN_PATH, has_labels=True)
eval_df = load_data(EVAL_PATH, has_labels=False)

print(f"Train: {len(train_df)} samples")
print(f"Eval: {len(eval_df)} samples (unlabeled)")

print("\nTrain label distribution:")
print(train_df['label'].value_counts())

#%% ============== METRICS ==============
def compute_metrics(eval_pred):
    """Compute metrics for training evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average='macro')
    weighted_f1 = f1_score(labels, predictions, average='weighted')
    accuracy = accuracy_score(labels, predictions)
    per_class_f1 = f1_score(labels, predictions, average=None)

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'f1_pro': per_class_f1[0],
        'f1_against': per_class_f1[1],
        'f1_neutral': per_class_f1[2],
    }

Loading data...
Train: 843 samples
Eval: 181 samples (unlabeled)

Train label distribution:
label
against    298
pro        286
neutral    259
Name: count, dtype: int64


In [ ]:
#%% ============== LOAD MODEL & TOKENIZER ==============
print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)


Loading model: aubmindlab/bert-base-arabertv02


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#%% ============== CREATE DATASETS ==============
# Training dataset (with labels)
train_dataset = StanceDataset(
    texts=train_df['Sentence_norm'].values,
    topics=train_df['Topic_norm'].values,
    labels=train_df['label'].map(LABEL2ID).values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

# Evaluation dataset (without labels - for prediction only)
eval_dataset = StanceDataset(
    texts=eval_df['Sentence_norm'].values,
    topics=eval_df['Topic_norm'].values,
    labels=None,  # No labels
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

In [ ]:
#%% ============== TRAINING ==============
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="no", #since by this step eval set is unlabeled
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)

print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)
trainer.train()


STARTING TRAINING


Step,Training Loss
10,1.331500
20,1.169000
30,1.042400
40,0.960000
50,0.925200
60,0.827600
70,0.824900
80,0.714600
90,0.726400
100,0.642100


TrainOutput(global_step=265, training_loss=0.5909203835253446, metrics={'train_runtime': 124.8578, 'train_samples_per_second': 33.758, 'train_steps_per_second': 2.122, 'total_flos': 277255763930880.0, 'train_loss': 0.5909203835253446, 'epoch': 5.0})

In [ ]:
#%% ============== SAVE MODEL ==============
print(f"\nSaving model to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

#%% ============== PREDICT ON EVALUATION SET ==============
print("\n" + "="*50)
print("PREDICTING ON EVALUATION SET")
print("="*50)

# Make predictions
eval_predictions = trainer.predict(eval_dataset)
eval_preds = np.argmax(eval_predictions.predictions, axis=1)

# Convert predictions to labels
predicted_labels = [ID2LABEL[pred] for pred in eval_preds]

# Add predictions to dataframe
eval_df['prediction'] = predicted_labels

print(f"\nPrediction distribution:")
print(pd.Series(predicted_labels).value_counts())

#%% ============== SAVE PREDICTIONS ==============
# Save as CSV
print(f"\nSaving predictions to {PREDICTIONS_CSV}")
eval_df.to_csv(PREDICTIONS_CSV, index=False)

# Create ZIP archive
print(f"Creating ZIP archive: {PREDICTIONS_ZIP}")
with zipfile.ZipFile(PREDICTIONS_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTIONS_CSV, os.path.basename(PREDICTIONS_CSV))

print("\n" + "="*50)
print("COMPLETE!")
print("="*50)
print(f"✓ CSV file saved: {PREDICTIONS_CSV}")
print(f"✓ ZIP archive saved: {PREDICTIONS_ZIP}")
print(f"✓ Total predictions: {len(eval_df)}")
print("\nPrediction summary:")
print(eval_df['prediction'].value_counts())



Saving model to ./stance_model

PREDICTING ON EVALUATION SET



Prediction distribution:
against    71
pro        60
neutral    50
Name: count, dtype: int64

Saving predictions to predictions.csv
Creating ZIP archive: predictions.zip

COMPLETE!
✓ CSV file saved: predictions.csv
✓ ZIP archive saved: predictions.zip
✓ Total predictions: 181

Prediction summary:
prediction
against    71
pro        60
neutral    50
Name: count, dtype: int64


In [ ]:
files.download("predictions.zip")
print(f"✓ ZIP archive downloaded:")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ ZIP archive downloaded:
